In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model

# load model
model = load_model("model_gru.h5")

model.summary()

2026-06-02 09:35:32.256157: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-02 09:35:32.258391: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-06-02 09:35:32.288884: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-02 09:35:32.288934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-02 09:35:32.290409: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 layer1 (GRU)                (None, 20)                1680      
                                                                 
 layer3 (Dense)              (None, 64)                1344      
                                                                 
 relu_0 (Activation)         (None, 64)                0         
                                                                 
 output_sigmoid (Dense)      (None, 1)                 65        
                                                                 
Total params: 3089 (12.07 KB)
Trainable params: 3089 (12.07 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [3]:
import numpy as np

# input data
data = np.load('x_test.npy')

# first entry
entry = data[5969]

print(entry)

[[1.31108925e-01 3.84247869e-01 5.03955424e-01 1.32139802e-01
  8.48756172e-04 5.29385149e-01]
 [1.08765453e-01 3.86480123e-01 5.01871824e-01 1.09554023e-01
  2.35831644e-03 5.25090396e-01]
 [9.33585465e-02 3.84385318e-01 5.00745356e-01 9.40175131e-02
  8.91348638e-04 5.29385149e-01]
 [6.27450645e-02 3.83578867e-01 5.03291845e-01 6.32293522e-02
  7.50077365e-04 4.52305615e-01]
 [6.26925603e-02 3.86411458e-01 5.03476977e-01 6.31722957e-02
  2.35040649e-03 5.25090396e-01]
 [5.45275509e-02 3.84845465e-01 5.00007629e-01 5.49023263e-02
  1.43604656e-03 5.25090396e-01]
 [5.16731404e-02 3.84776771e-01 5.01612663e-01 5.20477816e-02
  7.44137680e-04 5.25090396e-01]
 [5.06004281e-02 3.85984927e-01 5.02877772e-01 5.09809628e-02
  1.86477171e-03 5.47694385e-01]
 [4.29048873e-02 3.82153064e-01 5.02828658e-01 4.32332009e-02
  2.01745518e-03 5.25090396e-01]
 [4.23204936e-02 3.84130001e-01 5.00820041e-01 4.26232032e-02
  8.18970322e-04 5.72558761e-01]
 [3.98590378e-02 3.82499695e-01 5.06539524e-01 4.0

In [4]:
import numpy as np

# bin = np.binary_repr(num, width=width)

# converts decimal number as shows up in modelsim to actual decimal number
# since verilog implementation uses fixed point and python does not
def fixed_to_dec(num, nfrac):
    return num / (2 ** nfrac)

# vice versa
def dec_to_fixed(num, nfrac):
    return num * (2 ** nfrac)

In [5]:
import tensorflow as tf

# grab GRU layer from overall model
gru = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("layer1").output
)

# constant -0.0009765625 input
x = np.full((1, 20, 6), -0.0009765625, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for -1 input:", gru_output)

# constant 0 input
x = np.full((1, 20, 6), 0, dtype=np.float32)

gru_output = gru.predict(x)

print("GRU output shape:", gru_output.shape)
print("GRU output for 0 input:", gru_output)

# real input

x = np.expand_dims(entry, axis=0)

gru_output = gru.predict(x)
print("GRU output for actual input:", gru_output)

1/1 [==============================] - 0s 227ms/step
GRU output shape: (1, 20)
GRU output for -1 input: [[ 5.0905389e-03  1.6728804e-01  6.4357883e-01  2.5470734e-02
  -4.3091021e-02 -2.0401792e-05 -1.6998371e-02 -7.2253215e-01
   8.5067934e-01  2.3964050e-01 -1.8604267e-01  6.3019526e-01
   7.2923297e-01  9.4031227e-01 -3.4397484e-03 -3.2461387e-01
   1.3428175e-01  2.7717035e-02  4.5356870e-01  2.3317397e-01]]
1/1 [==============================] - 0s 17ms/step
GRU output shape: (1, 20)
GRU output for 0 input: [[ 0.00638667  0.16483407  0.6490506   0.01878716 -0.0463143  -0.00479441
  -0.0136489  -0.7286906   0.85105824  0.2505707  -0.17842358  0.63234377
   0.7343067   0.93981105 -0.00605593 -0.32525256  0.13518424  0.02427531
   0.45254812  0.23819612]]
1/1 [==============================] - 0s 14ms/step
GRU output for actual input: [[-0.13830498  0.1585753  -0.3846001   0.08903046  0.38578692  0.03404496
  -0.19405721  0.31036425 -0.1299223  -0.29487348  0.01821575 -0.34438294
  -

In [6]:
# manually run through GRU computation

import numpy as np

# =======================
# set config
# =======================
TIMESTEPS = 20
INPUT_SIZE = 6
HIDDEN_SIZE = 20

# =======================
# helper functions
# =======================
def load_vector(filename):
    return np.loadtxt(filename)

def load_matrix(filename, rows, cols):
    data = np.loadtxt(filename)
    return data.reshape(rows, cols)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# =======================
# get GRU weights
# =======================

gru = model.layers[0]

kernel, recurrent_kernel, bias = gru.get_weights()

# split kernels
W_z, W_r, W_h = np.split(kernel, 3, axis=1)
U_z, U_r, U_h = np.split(recurrent_kernel, 3, axis=1)

# split bias into input and recurrent
b_input = bias[0]   # (360,)
b_rec   = bias[1]   # (360,)

# split per gate (update, reset, candidate hidden state)
b_z, b_r, b_h = np.split(b_input, 3)
b_z_rec, b_r_rec, b_h_rec = np.split(b_rec, 3)

# =======================
# set input
# =======================
x = np.expand_dims(entry, axis=0)

# =======================
# initialize hidden state
# =======================
h = np.zeros(HIDDEN_SIZE) # more zeroes

# trace storage
z_trace = []
z_raw_trace = []
r_trace = []
r_raw_trace = []
h_tilde_trace = []
h_tilde_raw_trace = []
h_tilde_raw_U_trace = []
h_tilde_raw_W_trace = []
h_trace = []

# =======================
# GRU calculation
# =======================
for t in range(TIMESTEPS):
    x_t = x[0][t]

    # gates
    z_raw = x_t @ W_z + h @ U_z + b_z + b_z_rec
    r_raw = x_t @ W_r + h @ U_r + b_r + b_r_rec
    z = sigmoid(z_raw)
    r = sigmoid(r_raw)

    # candidate (reset_after=True)
    h_tilde_raw = x_t @ W_h + b_h + r * (h @ U_h + b_h_rec)
    # h_tilde_raw_U = h_t-1 * U_h + b_h_rec
    h_tilde_raw_U = h @ U_h + b_h_rec
    # h_tilde_raw_W = W_h * x_t + b_h
    h_tilde_raw_W = x_t @ W_h + b_h
    h_tilde = np.tanh(h_tilde_raw)

    # hidden state update
    h = (1 - z) * h_tilde + z * h

    # save traces
    z_trace.append(z)
    z_raw_trace.append(z_raw)
    r_trace.append(r)
    r_raw_trace.append(r_raw)
    h_tilde_trace.append(h_tilde)
    h_tilde_raw_trace.append(h_tilde_raw)
    h_tilde_raw_U_trace.append(h_tilde_raw_U)
    h_tilde_raw_W_trace.append(h_tilde_raw_W)
    h_trace.append(h)

# convert to arrays
z_raw_trace = np.array(z_raw_trace)
z_trace = np.array(z_trace)
r_raw_trace = np.array(r_raw_trace)
r_trace = np.array(r_trace)
h_tilde_trace = np.array(h_tilde_trace)
h_tilde_raw_trace = np.array(h_tilde_raw_trace)
h_tilde_raw_U_trace = np.array(h_tilde_raw_U_trace)
h_tilde_raw_W_trace = np.array(h_tilde_raw_W_trace)
h_trace = np.array(h_trace)

# =======================
# outputs
# =======================
print("final hidden state (first 10 units):")
print(h[:10])

print("\nhidden state evolution (first unit):")
print(h_trace[:, 0])

final hidden state (first 10 units):
[-0.13830505  0.15857533 -0.38460008  0.08903054  0.38578697  0.034045
 -0.19405723  0.31036433 -0.12992233 -0.29487354]

hidden state evolution (first unit):
[-0.00762032 -0.01352592 -0.02089499 -0.02978183 -0.03487692 -0.03750619
 -0.03886122 -0.03913665 -0.03956054 -0.03931051 -0.03968962 -0.03155625
 -0.03343484 -0.05272642 -0.06986421 -0.08399985 -0.09765951 -0.1107426
 -0.12466333 -0.13830505]


In [7]:
# print detailed evolution of what signals should look like in modelsim

print("evolution in modelsim (by timestep):\n")

num_steps = min(100, r_trace.shape[1])
nfrac = 10

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = dec_to_fixed(r_trace[t, :5], nfrac)
    z_fix = dec_to_fixed(z_trace[t, :5], nfrac)
    h_tilde_fix = dec_to_fixed(h_tilde_trace[t, :5], nfrac)
    h_fix = dec_to_fixed(h_trace[t, :5], nfrac)
    
    r_raw = dec_to_fixed(r_raw_trace[t, :5], nfrac)
    z_raw = dec_to_fixed(z_raw_trace[t, :5], nfrac)
    h_tilde_raw = dec_to_fixed(h_tilde_raw_trace[t, :5], nfrac)
    h_tilde_raw_U = dec_to_fixed(h_tilde_raw_U_trace[t, :5], nfrac)
    h_tilde_raw_W = dec_to_fixed(h_tilde_raw_W_trace[t, :5], nfrac)
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw_U : {h_tilde_raw_U}")
    print(f"  h_tilde_raw_W : {h_tilde_raw_W}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution in modelsim (by timestep):

t = 0
  r_raw   : [-122.75570078 -193.36198028  -62.46349164   53.22230575  217.51360224]
  r       : [481.34777431 463.80263351 496.38896743 525.30258195 566.17485433]
  z_raw   : [  579.4442783  -1175.58053055   455.19512367  -364.22118253
   445.98416817]
  z       : [653.11556075 246.62987414 623.96116183 421.89267613 621.76639246]
  h_tilde_raw_U : [-28.81407356  27.45491028  71.3564682   68.62835693 -78.93804932]
  h_tilde_raw_W : [  -8.00306226   72.48505655 -339.67782205  -18.4359996   296.32752994]
  h_tilde_raw : [ -21.54758392   84.9202711  -305.08742794   16.76971631  252.68227742]
  h_tilde : [ -21.54440414   84.72612891 -296.36966252   16.76821728  247.675536  ]
  h       : [  -7.80320727   64.31988428 -115.7806401     9.85963519   97.28850034]

t = 1
  r_raw   : [ -62.10007653 -261.76516899  -15.88023858 -102.17893387 -476.97409944]
  r       : [496.47973724 446.91275882 508.03001992 486.47644092 394.86667559]
  z_raw   : [  413.3976

In [8]:
# print detailed evolution of what signals should look like

print("evolution (by timestep):\n")

num_steps = min(15, r_trace.shape[1])
nfrac = 10

for t in range(num_steps):
    print(f"t = {t}")
    
    r_fix = r_trace[t, :5]
    z_fix = z_trace[t, :5]
    h_tilde_fix = h_tilde_trace[t, :5]
    h_fix = h_trace[t, :5]
    
    r_raw = r_raw_trace[t, :5]
    z_raw = z_raw_trace[t, :5]
    h_tilde_raw = h_tilde_raw_trace[t, :5]
    
    print(f"  r_raw   : {r_raw}")
    print(f"  r       : {r_fix}")
    
    print(f"  z_raw   : {z_raw}")
    print(f"  z       : {z_fix}")
    
    print(f"  h_tilde_raw : {h_tilde_raw}")
    print(f"  h_tilde : {h_tilde_fix}")
    
    print(f"  h       : {h_fix}")
    print()



evolution (by timestep):

t = 0
  r_raw   : [-0.11987861 -0.18883006 -0.0609995   0.05197491  0.21241563]
  r       : [0.47006619 0.45293226 0.48475485 0.5129908  0.55290513]
  z_raw   : [ 0.56586355 -1.14802786  0.44452649 -0.35568475  0.43553141]
  z       : [0.63780816 0.24084949 0.60933707 0.41200457 0.60719374]
  h_tilde_raw : [-0.02104256  0.08292995 -0.29793694  0.01637668  0.24676004]
  h_tilde : [-0.02103946  0.08274036 -0.2894235   0.01637521  0.24187064]
  h       : [-0.00762032  0.06281239 -0.11306703  0.00962855  0.0950083 ]

t = 1
  r_raw   : [-0.06064461 -0.25563005 -0.01550805 -0.09978412 -0.46579502]
  r       : [0.48484349 0.43643824 0.49612307 0.47507465 0.38561199]
  z_raw   : [ 0.40370866 -2.09351311  0.63270881 -0.2575956   0.43190923]
  z       : [0.59957838 0.10972891 0.65310342 0.43595485 0.60632948]
  h_tilde_raw : [-0.02237251  0.07095933 -0.27802386  0.0379477   0.23809932]
  h_tilde : [-0.02236878  0.07084047 -0.27107513  0.03792949  0.23369968]
  h       :

In [9]:
# print(np.shape(first_entry))

# predictions = model.predict(data)
# print(predictions)

# import pandas as pd

# # 1. Get predictions from your model
# predictions = model.predict(data)

# # 2. Convert to a DataFrame
# # If predictions is a simple array, use columns=['Column_Name']
# df = pd.DataFrame(predictions, columns=['Predicted_Labels'])

# # 3. Save to CSV
# df.to_csv('predictions.csv', index=False)

In [1]:
import numpy as np

y = np.load('y_test.npy')
print(y[0:211])

[[1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [0.]
 [1.]
 [1.]
 [1.]
 [0.]
 [0.]
 [1.

In [1]:
from sklearn.metrics import accuracy_score
import numpy as np

def accuracy_test():
    """
    Runs an accuracy test with bit precision (WIDTH, NINT)\n
    acc: tuple representing bit precision  (WIDTH, NINT)\n
    y_test: the correct values for the inputs
    """

    res_file = "handmade_toptag_gru_print_16_6_results.csv"

    res =  np.loadtxt(res_file, delimiter=",")
    res = res.reshape(-1, 1)
    y_test = np.load('y_test.npy')
    print(res)
    print(y_test)
    sample_count = min(len(y_test), len(res))
    print(sample_count)
    if sample_count == 0:
        raise RuntimeError(f"No comparable samples for {acc}: y_test={len(y_test)}, res={len(res)}")
    if len(res) != len(y_test):
        print(f"[WARN] {acc} produced {len(res)} results for {len(y_test)} labels; scoring first {sample_count} samples")
    acc_res= accuracy_score((y_test[0:sample_count]), np.round(res[0:sample_count]))
    print(acc_res)

    return acc_res

accuracy_test()

[[0.95019531]
 [0.89550781]
 [0.73242188]
 ...
 [0.57519531]
 [0.04394531]
 [0.42871094]]
[[1.]
 [1.]
 [1.]
 ...
 [1.]
 [0.]
 [1.]]
19951
0.8182046012731191


0.8182046012731191